# NumpyDataset: ограниченный LRU memory cache

## Goal

Показать четвертый этап реализации датасета: быстрый процесс-локальный кэш загруженных `LoadedSample` в RAM. Ноутбук демонстрирует:

- порядок `RAM → disk → FIF` в режиме `both`;
- лимит RAM в байтах;
- LRU-вытеснение;
- очистку кэша;
- поведение семпла, который больше заданного лимита.

## Setup

Демонстрация использует три реальных блока `Data_Train/S_1/Trial_1` и отдельный ignored disk cache в `artifacts/cache/notebook-memory-demo`.

In [1]:
import shutil
import sys
import time
from pathlib import Path
from unittest.mock import patch

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Project root with pyproject.toml was not found")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.datasets import NumpyDataset

DATASET_DIR = PROJECT_ROOT / "data" / "Data_Train"
DEMO_CACHE_DIR = PROJECT_ROOT / "artifacts" / "cache" / "notebook-memory-demo"
SAMPLE_KEYS = [(1, 1, block_index) for block_index in (1, 2, 3)]

shutil.rmtree(DEMO_CACHE_DIR, ignore_errors=True)
print(f"Dataset source: {DATASET_DIR}")
print(f"Demo disk cache: {DEMO_CACHE_DIR}")

Dataset source: /home/slauva/Projects/master-thesis-2024-2026/code/data/Data_Train
Demo disk cache: /home/slauva/Projects/master-thesis-2024-2026/code/artifacts/cache/notebook-memory-demo


### Key Assumptions

- Memory cache принадлежит одному объекту и одному процессу.
- Размер entry равен `eeg.nbytes + eog.nbytes`.
- RAM entry также инвалидируется при изменении `size` или `mtime_ns` исходного FIF.
- Массивы, превышающие лимит целиком, возвращаются пользователю, но не сохраняются в RAM.

## Steps

### 1. Создать режим `both`

Лимит `9 MiB` позволяет хранить примерно два текущих `float32`-семпла по `4.15 MiB`.

In [2]:
memory_limit_bytes = 9 * 2**20
dataset = NumpyDataset(
    DATASET_DIR,
    cache_policy="both",
    cache_dir=DEMO_CACHE_DIR,
    memory_cache_bytes=memory_limit_bytes,
)

pd.Series(
    {
        "cache_policy": dataset.cache_policy,
        "memory_limit_mib": dataset.memory_cache_bytes / 2**20,
        "memory_items": dataset.memory_cache_items,
        "memory_used_mib": dataset.memory_cache_current_bytes / 2**20,
    },
    name="configuration",
)

cache_policy        both
memory_limit_mib     9.0
memory_items           0
memory_used_mib      0.0
Name: configuration, dtype: object

### 2. Первый доступ: FIF miss

Семпл загружается через MNE, записывается на disk и помещается в RAM.

In [3]:
started_at = time.perf_counter()
first_loaded = dataset[SAMPLE_KEYS[0]]
fif_load_seconds = time.perf_counter() - started_at
sample_size_bytes = first_loaded.eeg.nbytes + first_loaded.eog.nbytes

pd.Series(
    {
        "sample_size_mib": sample_size_bytes / 2**20,
        "load_seconds": fif_load_seconds,
        "memory_keys": dataset.memory_cache_keys,
        "disk_entry_exists": dataset.get_cache_entry_path(SAMPLE_KEYS[0]).is_dir(),
    },
    name="first_load",
)

sample_size_mib           4.15065
load_seconds             0.028297
memory_keys          ((1, 1, 1),)
disk_entry_exists            True
Name: first_load, dtype: object

### 3. Второй доступ: доказать RAM hit

На время чтения запрещаем и disk cache, и FIF loader. Возвращается тот же объект из RAM.

In [4]:
with (
    patch.object(dataset, "_load_disk_cache", side_effect=AssertionError("Unexpected disk read")),
    patch.object(dataset, "_load_sample", side_effect=AssertionError("Unexpected FIF read")),
):
    started_at = time.perf_counter()
    memory_loaded = dataset[SAMPLE_KEYS[0]]
    memory_load_seconds = time.perf_counter() - started_at

assert memory_loaded is first_loaded
pd.DataFrame(
    {
        "source": ["FIF miss", "RAM hit"],
        "seconds": [fif_load_seconds, memory_load_seconds],
    }
)

,source,seconds
0,FIF miss,0.028297
1,RAM hit,0.000139


### 4. Очистить RAM и проверить disk fallback

После `clear_memory_cache()` запрещаем MNE-чтение. Данные восстанавливаются из `.npy`, затем снова попадают в RAM.

In [5]:
dataset.clear_memory_cache()
assert dataset.memory_cache_items == 0

with patch("mne.io.read_raw_fif", side_effect=AssertionError("Unexpected FIF read")):
    disk_loaded = dataset[SAMPLE_KEYS[0]]

np.testing.assert_array_equal(disk_loaded.eeg, first_loaded.eeg)
np.testing.assert_array_equal(disk_loaded.eog, first_loaded.eog)

pd.Series(
    {
        "memory_items_after_disk_load": dataset.memory_cache_items,
        "memory_keys": dataset.memory_cache_keys,
        "arrays_equal": True,
    },
    name="disk_fallback",
)

memory_items_after_disk_load               1
memory_keys                     ((1, 1, 1),)
arrays_equal                            True
Name: disk_fallback, dtype: object

### 5. Продемонстрировать LRU-вытеснение

Загружаем блоки `1, 2`, повторно обращаемся к `1`, затем загружаем `3`. Наименее недавно использованный блок `2` должен быть вытеснен.

In [6]:
dataset.clear_memory_cache()
lru_steps = []

for key in (SAMPLE_KEYS[0], SAMPLE_KEYS[1], SAMPLE_KEYS[0], SAMPLE_KEYS[2]):
    dataset[key]
    lru_steps.append(
        {
            "requested": key,
            "lru_to_mru": dataset.memory_cache_keys,
            "items": dataset.memory_cache_items,
            "used_mib": dataset.memory_cache_current_bytes / 2**20,
        }
    )

display(pd.DataFrame(lru_steps))

,requested,lru_to_mru,items,used_mib
0,"(1, 1, 1)","((1, 1, 1),)",1,4.15065
1,"(1, 1, 2)","((1, 1, 1), (1, 1, 2))",2,8.30130
2,"(1, 1, 1)","((1, 1, 2), (1, 1, 1))",2,8.30130
3,"(1, 1, 3)","((1, 1, 1), (1, 1, 3))",2,8.30130


### 6. Семпл больше лимита

При лимите `1 byte` данные загружаются, но RAM cache остается пустым.

In [7]:
tiny_cache_dataset = NumpyDataset(
    DATASET_DIR,
    cache_policy="memory",
    memory_cache_bytes=1,
)
oversized_loaded = tiny_cache_dataset[SAMPLE_KEYS[0]]

pd.Series(
    {
        "loaded_shape": oversized_loaded.eeg.shape,
        "memory_items": tiny_cache_dataset.memory_cache_items,
        "memory_used_bytes": tiny_cache_dataset.memory_cache_current_bytes,
    },
    name="oversized_sample",
)

loaded_shape         (63, 16001)
memory_items                   0
memory_used_bytes              0
Name: oversized_sample, dtype: object

## Checks

Проверяем порядок уровней и итог LRU.

In [8]:
assert memory_loaded is first_loaded
assert dataset.memory_cache_keys == (SAMPLE_KEYS[0], SAMPLE_KEYS[2])
assert dataset.memory_cache_items == 2
assert dataset.memory_cache_current_bytes <= dataset.memory_cache_bytes
assert SAMPLE_KEYS[1] not in dataset.memory_cache_keys
assert tiny_cache_dataset.memory_cache_items == 0
assert tiny_cache_dataset.memory_cache_current_bytes == 0

pd.Series(
    {
        "ram_before_disk": "passed",
        "disk_before_fif": "passed",
        "lru_eviction": "passed",
        "byte_limit": "passed",
        "oversized_sample": "passed",
    },
    name="validation",
)

ram_before_disk     passed
disk_before_fif     passed
lru_eviction        passed
byte_limit          passed
oversized_sample    passed
Name: validation, dtype: str

## Next Steps

Memory cache ускоряет повторные обращения внутри одного процесса. Следующий этап добавит явный `warm_cache()` с multiprocessing для предварительного построения disk cache; RAM между worker-процессами разделяться не будет.